In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


In [ ]:
pip install SpeechRecognition openai-whisper torch transformers faiss-cpu python-dotenv fastapi uvicorn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a7f36ca5b1261affecf2f452884f711253866cbe5b7a46c443e8a4ea16090ed8
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import whisper

# Load model
model = whisper.load_model("base")

# Record or upload audio file (e.g., claim.mp3)
result = model.transcribe("/content/33eac7d0-0eda-4bbc-90d3-6579720ef0d4.wav")
transcribed_text = result["text"]
print("Transcribed:", transcribed_text)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed:  Good morning. I would like to report a motor vehicle accident that occurred on 25 May 2026 at approximately 6 p.m. near Santons City in Johannesburg. I was driving my Toyota Corolla when another vehicle failed to stop at a red traffic light and collided with the rear of my vehicle. The rear bumper, boot, and left-tail light sustained visible damage. No injuries were reported at the scene. The police attended the scene and a case number was issued. I have comprehensive insurance cover and would like to submit a claim for repairs. Thank you.


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(transcribed_text, return_tensors="pt", padding=True, truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state  # Shape: [1, seq_len, 768]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# knowledge_base.py
insurance_knowledge = [
    "Car accident claims require date, location, fault, vehicle damage.",
    "Third-party fault means the other driver was responsible.",
    "Claims must be filed within 30 days of incident.",
    "Sandton is in Gauteng, South Africa.",
    "Hit-and-run: driver fled the scene."
]


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
knowledge_embeddings = embedding_model.encode(insurance_knowledge)
knowledge_embeddings = np.array(knowledge_embeddings).astype("float32")

# Build FAISS index
index = faiss.IndexFlatL2(knowledge_embeddings.shape[1])
index.add(knowledge_embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Note on Groq Models

Groq models can be decommissioned over time. If you encounter errors similar to `model_decommissioned`, please refer to the [Groq documentation](https://console.groq.com/docs/deprecations) for the most up-to-date list of supported models.

In [ ]:
query_embedding = embedding_model.encode([transcribed_text])
query_embedding = np.array(query_embedding).astype("float32")

# Search
D, I = index.search(query_embedding, k=2)  # Get 2 closest rules
retrieved_rules = [insurance_knowledge[i] for i in I[0]]


In [ ]:
import os
from groq import Groq

client = Groq(
    api_key=" Groq API Key " # Pass API key directly
)

prompt = f"""
You are an insurance claims assistant.

Use the following context to help structure the claim.

Context:
{retrieved_rules}

Claim narration:
"{transcribed_text}"

Return ONLY valid JSON in the following format:

{{
    "claim_type": "",
    "date": "",
    "location": "",
    "fault": "",
    "description": ""
}}
"""

chat_completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

structured_output = chat_completion.choices[0].message.content

print(structured_output)

{
    "claim_type": "Car accident",
    "date": "25 May 2026",
    "location": "Santons City, Johannesburg",
    "fault": "Not at fault",
    "description": "Rear bumper, boot, and left-tail light damage due to collision from another vehicle that failed to stop at a red traffic light"
}


In [ ]:
from fastapi import FastAPI, File, UploadFile
import whisper
import json

app = FastAPI()
model = whisper.load_model("base")

@app.post("/submit-claim/")
async def submit_claim(audio: UploadFile = File(...)):
    with open("temp.mp3", "wb") as f:
        f.write(await audio.read())

    result = model.transcribe("temp.mp3")
    text = result["text"]

    # (Insert RAG + LLM steps here)
    # For demo, mock structured output
    structured_claim = {
        "claim_type": "car accident",
        "date": "yesterday",
        "location": "Sandton",
        "fault": "third-party",
        "description": text
    }

    return {"transcribed": text, "structured": structured_claim}


In [ ]:
%%writefile main.py
from fastapi import FastAPI, File, UploadFile
import whisper
import json

app = FastAPI()
model = whisper.load_model("base")

@app.post("/submit-claim/")
async def submit_claim(audio: UploadFile = File(...)):
    with open("temp.mp3", "wb") as f:
        f.write(await audio.read())

    result = model.transcribe("temp.mp3")
    text = result["text"]

    # (Insert RAG + LLM steps here)
    # For demo, mock structured output
    structured_claim = {
        "claim_type": "car accident",
        "date": "yesterday",
        "location": "Sandton",
        "fault": "third-party",
        "description": text
    }

    return {"transcribed": text, "structured": structured_claim}

Writing main.py


In [ ]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [6037] using WatchFiles
INFO:     Started server process [6060]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [6060]
INFO:     Stopping reloader process [6037]
